In [1]:
import os
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader, Subset
import torch.optim as optim
from torchvision.utils import make_grid,save_image
from torchvision import transforms
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report
from PIL import Image
import matplotlib.pyplot as plt
import timeit
from tqdm import tqdm
import seaborn as sns
import altair as alt
import random
from torch.cuda.amp import GradScaler, autocast
from transformers import AutoImageProcessor, AutoModel


import warnings
warnings.filterwarnings("ignore")

In [2]:
class LoadDataset(Dataset):
    def __init__(self, root_dir, dataset_type, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.images = []
        self.labels = []

        for sub_dataset in os.listdir(root_dir):
            sub_dataset_path = os.path.join(root_dir, sub_dataset)
            for nasted_dir in os.listdir(sub_dataset_path):
                nasted_dir_path = os.path.join(sub_dataset_path, nasted_dir)
                dataset_type_dir = os.path.join(nasted_dir_path, dataset_type)
                for label in os.listdir(dataset_type_dir):
                    label_folder = os.path.join(dataset_type_dir, label)
                    for image_name in os.listdir(label_folder):
                        image_path = os.path.join(label_folder, image_name)
                        self.images.append(image_path)
                        self.labels.append(label)

        self.label_encoder = LabelEncoder()
        self.encoded_labels = self.label_encoder.fit_transform(self.labels)

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img_path = self.images[idx]
        label = self.encoded_labels[idx]

        image = Image.open(img_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, label

    def get_label_mapping(self):
        return dict(zip(self.label_encoder.classes_, range(len(self.label_encoder.classes_))))

In [3]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(
        (0.5, 0.5, 0.5),
        (0.5, 0.5, 0.5)
    ),
])

In [4]:
dataset_path = '/kaggle/input/real-and-fake-images-dataset-for-image-forensics'

# Load datasets
train_dataset = LoadDataset(root_dir=dataset_path, dataset_type="train", transform=transform)
val_dataset = LoadDataset(root_dir=dataset_path, dataset_type="validation", transform=transform)
test_dataset = LoadDataset(root_dir=dataset_path, dataset_type="test", transform=transform)

print(f"Total Training Dataset: {len(train_dataset)}")
print(f"Total Validation Dataset: {len(val_dataset)}")
print(f"Total Test Dataset: {len(test_dataset)}")
print(f"Total image samples {len(train_dataset) + len(val_dataset) + len(test_dataset)}")

Total Training Dataset: 160002
Total Validation Dataset: 49428
Total Test Dataset: 20905
Total image samples 230335


In [5]:
# Define data loaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Total Batches for Training {len(train_loader)}")

Total Batches for Training 5001


In [6]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [7]:
os.makedirs("generated_images", exist_ok=True)

# Initialize and configure the training process

In [8]:
class Generator(nn.Module):
    def __init__(self, z_dim=100):
        super(Generator, self).__init__()
        self.z_dim = z_dim
        self.main = nn.Sequential(
            # First layer - z_dim -> 512 (first 4x4 image)
            nn.ConvTranspose2d(z_dim, 512, 4, 1, 0, bias=False),
            nn.BatchNorm2d(512),
            nn.ReLU(True),
            
            # Second layer - 512 -> 256 (8x8 image)
            nn.ConvTranspose2d(512, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            
            # Third layer - 256 -> 128 (16x16 image)
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            # Fourth layer - 128 -> 64 (32x32 image)
            nn.ConvTranspose2d(128, 64, 4, 2, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(True),
            
            # Fifth layer - 64 -> 32 (64x64 image)
            nn.ConvTranspose2d(64, 32, 4, 2, 1, bias=False),
            nn.BatchNorm2d(32),
            nn.ReLU(True),
            
            # Sixth layer - 32 -> 16 (128x128 image)
            nn.ConvTranspose2d(32, 16, 4, 2, 1, bias=False),
            nn.BatchNorm2d(16),
            nn.ReLU(True),
            
            # Seventh layer - 16 -> 3 (256x256 image)
            nn.ConvTranspose2d(16, 3, 4, 2, 1, bias=False),
            nn.Tanh()  # Output a 3-channel image (RGB)
        )

    def forward(self, input):
        return self.main(input)

In [9]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.main = nn.Sequential(
            
            # First layer - 3 -> 64 (128x128 image)
            nn.Conv2d(3, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Second layer - 64 -> 128 (64x64 image)
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Third layer - 128 -> 256 (32x32 image)
            nn.Conv2d(128, 256, 4, 2, 1, bias=False),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Fourth layer - 256 -> 512 (16x16 image)
            nn.Conv2d(256, 512, 4, 2, 1, bias=False),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Fifth layer - 512 -> 1024 (8x8 image)
            nn.Conv2d(512, 1024, 4, 2, 1, bias=False),
            nn.BatchNorm2d(1024),
            nn.LeakyReLU(0.2, inplace=True),
            
            # Final layer - 1024 -> 1 (output 1x1)
            nn.Conv2d(1024, 1, 8, 1, 0, bias=False),
        )

    def forward(self, input):
        return self.main(input)

In [10]:
z_dim   = 100
LR      = 0.0002
BETAS   = (0.5, 0.999)
use_amp = torch.cuda.is_available()

def weights_init(m):
    classname = m.__class__.__name__
    if "Conv" in classname:
        nn.init.normal_(m.weight.data, 0.0, 0.02)
    elif "BatchNorm" in classname:
        nn.init.normal_(m.weight.data, 1.0, 0.02)
        nn.init.constant_(m.bias.data, 0)

In [11]:
netG = Generator(z_dim).to(device)
netD = Discriminator().to(device)

netG.apply(weights_init)
netD.apply(weights_init)

criterion = nn.BCEWithLogitsLoss().to(device)

optimizerG = optim.Adam(netG.parameters(), lr=LR, betas=BETAS)
optimizerD = optim.Adam(netD.parameters(), lr=LR, betas=BETAS)

schedulerG = optim.lr_scheduler.StepLR(optimizerG, step_size=20, gamma=0.5)
schedulerD = optim.lr_scheduler.StepLR(optimizerD, step_size=20, gamma=0.5)

scalerG = torch.amp.GradScaler("cuda", enabled=use_amp)
scalerD = torch.amp.GradScaler("cuda", enabled=use_amp)

REAL_LABEL = 0.9 
FAKE_LABEL = 0.0

epochs = 2

loss_dict = {
    "epoch": [],
    "train": [],  # Discriminator training loss
    "val": [],    # Discriminator validation loss
    "G": []       # Generator training loss
}

accuracy_dict = {
    "epoch": [],
    "train": [],       # Overall discriminator training accuracy
    "val": [],         # Overall discriminator validation accuracy
    "train_real": [],  # Training accuracy on real images
    "train_fake": [],  # Training accuracy on generated images
    "val_real": [],    # Validation accuracy on real images
    "val_fake": []     # Validation accuracy on generated images
}

patience = 5
min_delta = 0.001
best_val_accuracy = 0.0
counter = 0
early_stop = False

## Begin the training loop

In [12]:
fixed_noise = torch.randn(64, z_dim, 1, 1, device=device)

start = timeit.default_timer()

for epoch in range(epochs):

    epoch_start = timeit.default_timer()

    netG.train()
    netD.train()

    train_loss = 0
    train_loss_G = 0
    train_preds, train_labels = [], []

    for batch_idx, batch in enumerate(train_loader):

        images, labels = batch
        images = images.to(device)

        b = images.size(0)
        noise = torch.randn(b, z_dim, 1, 1, device=device)

        # Train Discriminator
        optimizerD.zero_grad()

        with autocast(enabled=use_amp):
            fake = netG(noise)

            output_real = netD(images).view(-1)
            output_fake = netD(fake.detach()).view(-1)

            loss_D = criterion(
                output_real,
                torch.full((b,), REAL_LABEL, device=device)
            ) + criterion(
                output_fake,
                torch.full((b,), FAKE_LABEL, device=device)
            )

        scalerD.scale(loss_D).backward()
        scalerD.unscale_(optimizerD)
        torch.nn.utils.clip_grad_norm_(netD.parameters(), max_norm=1.0)
        scalerD.step(optimizerD)
        scalerD.update()

        # Train Generator
        optimizerG.zero_grad()

        with autocast(enabled=use_amp):
            output = netD(fake).view(-1)
            loss_G = criterion(output, torch.ones(b, device=device))

        scalerG.scale(loss_G).backward()
        scalerG.unscale_(optimizerG)
        torch.nn.utils.clip_grad_norm_(netG.parameters(), max_norm=1.0)
        scalerG.step(optimizerG)
        scalerG.update()

        train_loss += loss_D.item()
        train_loss_G += loss_G.item()

        # Calculate predictions
        train_preds.extend((output_real > 0).long().cpu().tolist())
        train_labels.extend([1] * b)

        train_preds.extend((output_fake > 0).long().cpu().tolist())
        train_labels.extend([0] * b)

    # Calculate training metrics
    train_loss /= len(train_loader)
    train_loss_G /= len(train_loader)

    train_accuracy = sum(1 for x, y in zip(train_preds, train_labels) if x == y) / len(train_labels)

    train_real_accuracy = sum(1 for x, y in zip(train_preds, train_labels) if y == 1 and x == 1) / train_labels.count(1)

    train_fake_accuracy = sum(1 for x, y in zip(train_preds, train_labels) if y == 0 and x == 0) / train_labels.count(0)

    # Validation
    netD.eval()

    val_loss = 0
    val_preds, val_labels = [], []

    with torch.no_grad():
        for images, labels in val_loader:
            
            images = images.to(device)
            labels = labels.to(device).float()

            with autocast(enabled=use_amp):
                logits = netD(images).view(-1)
                loss = criterion(logits, labels)

            val_loss += loss.item() * images.size(0)   # weight by batch size
            val_preds.extend((logits > 0).long().cpu().tolist())
            val_labels.extend(labels.long().cpu().tolist())


    # Calculate validation metrics
    val_loss /= len(val_loader)

    val_accuracy = sum(1 for x, y in zip(val_preds, val_labels) if x == y) / len(val_labels)

    val_real_accuracy = sum(1 for x, y in zip(val_preds, val_labels) if y == 1 and x == 1) / val_labels.count(1)

    val_fake_accuracy = sum(1 for x, y in zip(val_preds, val_labels) if y == 0 and x == 0) / val_labels.count(0)

    # Save training history
    loss_dict["epoch"].append(epoch + 1)
    loss_dict["train"].append(train_loss)
    loss_dict["val"].append(val_loss)
    loss_dict["G"].append(train_loss_G)

    accuracy_dict["epoch"].append(epoch + 1)
    accuracy_dict["train"].append(train_accuracy)
    accuracy_dict["val"].append(val_accuracy)
    accuracy_dict["train_real"].append(train_real_accuracy)
    accuracy_dict["train_fake"].append(train_fake_accuracy)
    accuracy_dict["val_real"].append(val_real_accuracy)
    accuracy_dict["val_fake"].append(val_fake_accuracy)

    # Update learning rate
    schedulerG.step()
    schedulerD.step()

    # Save generated images
    with torch.no_grad():
        with autocast(enabled=use_amp):
            sample_images = netG(fixed_noise)

        save_image(
            sample_images.detach().cpu(),
            f"generated_images/epoch_{epoch + 1:03d}.png",
            normalize=True,
            value_range=(-1, 1),
            nrow=8
        )

    # Save GAN checkpoint
    torch.save({
        "epoch": epoch + 1,

        "generator_state_dict": netG.state_dict(),
        "discriminator_state_dict": netD.state_dict(),

        "optimizerG_state_dict": optimizerG.state_dict(),
        "optimizerD_state_dict": optimizerD.state_dict(),

        "schedulerG_state_dict": schedulerG.state_dict(),
        "schedulerD_state_dict": schedulerD.state_dict(),

        "scalerG_state_dict": scalerG.state_dict(),
        "scalerD_state_dict": scalerD.state_dict(),

        "loss_dict": loss_dict,
        "accuracy_dict": accuracy_dict,

        "fixed_noise": fixed_noise,

    }, "gan_checkpoint.pth")

    epoch_end = timeit.default_timer()
    epoch_duration = epoch_end - epoch_start

    print(
        f"Epoch {epoch + 1}/{epochs}: "
        f"Train Loss: {train_loss:.4f}, "
        f"Val Loss: {val_loss:.4f}, "
        f"G Loss: {train_loss_G:.4f}, "
        f"Train Accuracy: {train_accuracy * 100:.2f}%, "
        f"Val Accuracy: {val_accuracy * 100:.2f}%, "
        f"Duration: {epoch_duration / 60:.2f} min"
    )

    print(
        f"    Train Real Acc: {train_real_accuracy * 100:.2f}%, "
        f"Train Fake Acc: {train_fake_accuracy * 100:.2f}%, "
        f"Val Real Acc: {val_real_accuracy * 100:.2f}%, "
        f"Val Fake Acc: {val_fake_accuracy * 100:.2f}%"
    )

    torch.cuda.empty_cache()

stop = timeit.default_timer()

print(f"Total Training Time: {(stop - start) / 60:.2f} min")

Epoch 1/2: Train Loss: 1.1216, Val Loss: 41.1651, G Loss: 7.3231, Train Accuracy: 84.61%, Val Accuracy: 50.72%, Duration: 275.28 min
    Train Real Acc: 80.67%, Train Fake Acc: 88.55%, Val Real Acc: 62.01%, Val Fake Acc: 39.36%
Epoch 2/2: Train Loss: 0.7461, Val Loss: 35.6522, G Loss: 7.6175, Train Accuracy: 91.91%, Val Accuracy: 45.98%, Duration: 62.92 min
    Train Real Acc: 87.25%, Train Fake Acc: 96.57%, Val Real Acc: 46.37%, Val Fake Acc: 45.58%
Total Training Time: 338.20 min


# Initializing Test

In [13]:
y_true = []
y_pred = []
y_scores = []

In [14]:
netD.eval()

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        # Get discriminator predictions
        outputs = netD(images)

        # Convert probabilities into binary predictions
        predictions = (outputs >= 0.5).float()

        # Store actual labels and predictions
        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

        # Store prediction probabilities
        y_scores.extend(outputs.cpu().numpy())

y_true = np.array(y_true).flatten()
y_pred = np.array(y_pred).flatten()
y_scores = np.array(y_scores).flatten()

print("Testing completed!")

Testing completed!


In [15]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report
)

print("Accuracy:", accuracy_score(y_true, y_pred))
print("Precision:", precision_score(y_true, y_pred))
print("Recall:", recall_score(y_true, y_pred))
print("F1 Score:", f1_score(y_true, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_true,
    y_pred,
    target_names=["Fake", "Real"]
))

print("\nConfusion Matrix:")
print(confusion_matrix(y_true, y_pred))

Accuracy: 0.4618512317627362
Precision: 0.45859305431878894
Recall: 0.44511668107173724
F1 Score: 0.4517543859649123

Classification Report:
              precision    recall  f1-score   support

        Fake       0.46      0.48      0.47     10492
        Real       0.46      0.45      0.45     10413

    accuracy                           0.46     20905
   macro avg       0.46      0.46      0.46     20905
weighted avg       0.46      0.46      0.46     20905


Confusion Matrix:
[[5020 5472]
 [5778 4635]]
